# Test — sous-échantillonnage équilibré (130K/130K) vs. `scale_pos_weight`

**Objectif** : trancher, sur des données identiques et une évaluation identique, si
l'approche "même effectif par classe" (comme testée par un ami avec un F1 ≈ 0,6)
apporte un vrai gain par rapport à la référence actuelle
(`pipeline_fast_tuning_v1_4_fixed.ipynb`, LightGBM, `scale_pos_weight`, F1 classe 1 =
0,229).

**Règle non négociable pour que la comparaison ait un sens** : le modèle est
entraîné sur un sous-échantillon équilibré, mais **toujours évalué sur `X_val`/`y_val`
réels, non touchés, avec le déséquilibre naturel (~4,25% de positifs)** — jamais sur
un jeu de test lui-même rééquilibré. C'est ce point précis qui peut faire qu'un F1 de
0,6 mesuré autrement ne soit pas comparable à notre 0,229.

Ce notebook part du principe que `pipeline_fast_tuning_v1_4_fixed.ipynb` a déjà tourné
au moins une fois avec `RECOLLECTER_XY=True` (section 9) — on **réutilise directement
son cache disque** (`./xy_cache/X_fit_f32.npy`, etc.) plutôt que de tout recollecter
depuis Spark. Aucune session Spark n'est nécessaire ici : c'est voulu, pour que ce
test reste rapide et léger en mémoire.

Deux variantes testées :

1. **Sous-échantillonnage simple** — un seul tirage 130K/130K (ou moins si moins de
   130K positifs sont disponibles côté fit), reproduisant directement ce que ton ami
   a fait.
2. **Balanced Bagging (EasyEnsemble)** — plusieurs modèles, chacun entraîné sur tous
   les positifs + un tirage *différent* de négatifs, puis moyennés. Ça répond au
   défaut principal de l'option 1 (jeter ~96% des négatifs) sans jamais charger plus
   de 130K lignes en mémoire à la fois — donc toujours memory-safe.


## 1. Setup

In [1]:
import os
import json as _json
import time

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, precision_recall_curve, classification_report, make_scorer
from lightgbm import LGBMClassifier

RANDOM_SEED = 42
XY_CACHE_DIR = "./xy_cache"                    # même convention que pipeline_fast_tuning_v1_4_fixed.ipynb, section 9
CHECKPOINT_DIR = "./models_checkpoint_undersampling"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Référence à battre -- LightGBM, scale_pos_weight, population complète.
# (pipeline_fast_tuning_v1_4_fixed.ipynb, section 11, comparaison_finale)
F1_BASELINE = 0.2288
SEUIL_BASELINE = 0.597

N_PAR_CLASSE_CIBLE = 130_000  # ce que ton ami a utilisé -- voir section 2 pour ce qui est réellement atteignable


## 2. Chargement du cache `X`/`y` (aucun recalcul Spark)

Si `FileNotFoundError` ici : retourner sur `pipeline_fast_tuning_v1_4_fixed.ipynb`,
section 9, mettre `RECOLLECTER_XY = True`, exécuter une fois (~10 min), puis relancer
ce notebook — il relira le même cache disque.


In [2]:
try:
    X_fit = np.load(f"{XY_CACHE_DIR}/X_fit_f32.npy", mmap_mode="r")
    y_fit = np.load(f"{XY_CACHE_DIR}/y_fit.npy", mmap_mode="r")
    X_val = np.load(f"{XY_CACHE_DIR}/X_val_f32.npy", mmap_mode="r")
    y_val = np.load(f"{XY_CACHE_DIR}/y_val.npy", mmap_mode="r")
except FileNotFoundError as e:
    raise RuntimeError(
        f"Aucun cache X/y trouvé dans '{XY_CACHE_DIR}'. Ce notebook réutilise le cache "
        f"déjà produit par pipeline_fast_tuning_v1_4_fixed.ipynb (section 9, "
        f"RECOLLECTER_XY=True) plutôt que de recollecter depuis Spark. "
        f"Générez-le une première fois là-bas, puis relancez ce notebook."
    ) from e

print(f"X_fit {X_fit.shape} {X_fit.dtype}, X_val {X_val.shape} {X_val.dtype}")
print(f"positifs fit={y_fit.mean():.3%}  (n={int((y_fit==1).sum())})  |  "
      f"positifs val={y_val.mean():.3%}  (n={int((y_val==1).sum())})")
print()
print("Rappel -- X_val/y_val NE SONT PAS MODIFIÉS dans ce notebook : c'est le même "
      "jeu de validation, avec le même déséquilibre naturel, que celui utilisé pour "
      "obtenir le F1 de référence ci-dessus. Toute comparaison se fait dessus, jamais "
      "sur un jeu de test rééquilibré.")


X_fit (2538213, 189) float32, X_val (634074, 189) float32
positifs fit=4.252%  (n=107916)  |  positifs val=4.260%  (n=27011)

Rappel -- X_val/y_val NE SONT PAS MODIFIÉS dans ce notebook : c'est le même jeu de validation, avec le même déséquilibre naturel, que celui utilisé pour obtenir le F1 de référence ci-dessus. Toute comparaison se fait dessus, jamais sur un jeu de test rééquilibré.


## 3. Construction du sous-échantillon équilibré

Tiré uniquement depuis `X_fit`/`y_fit` (jamais depuis `X_val`/`y_val`, qui doivent
rester intacts pour l'évaluation). Le nombre de positifs disponibles côté fit
(~80% de la population totale) est plus bas que 130K si le total plateforme est
d'environ 135K positifs sur l'ensemble train+val — dans ce cas `N_PAR_CLASSE_REEL`
sera plafonné automatiquement, affiché ci-dessous, plutôt que de piocher dans `X_val`
pour compléter (ce qui contaminerait l'évaluation).


In [3]:
rng = np.random.RandomState(RANDOM_SEED)

idx_pos_dispo = np.where(np.asarray(y_fit) == 1)[0]
idx_neg_dispo = np.where(np.asarray(y_fit) == 0)[0]

N_PAR_CLASSE_REEL = min(N_PAR_CLASSE_CIBLE, len(idx_pos_dispo))
if N_PAR_CLASSE_REEL < N_PAR_CLASSE_CIBLE:
    print(
        f"AVERTISSEMENT : seulement {len(idx_pos_dispo)} positifs disponibles côté fit "
        f"(< {N_PAR_CLASSE_CIBLE} demandés). N_PAR_CLASSE_REEL plafonné à "
        f"{N_PAR_CLASSE_REEL} -- on n'ira PAS chercher les positifs manquants dans "
        f"X_val, ça fausserait l'évaluation qui suit."
    )

idx_pos_echantillon = rng.choice(idx_pos_dispo, size=N_PAR_CLASSE_REEL, replace=False)
idx_neg_echantillon = rng.choice(idx_neg_dispo, size=N_PAR_CLASSE_REEL, replace=False)

idx_sous_echantillon = np.concatenate([idx_pos_echantillon, idx_neg_echantillon])
rng.shuffle(idx_sous_echantillon)

X_sous = np.asarray(X_fit[idx_sous_echantillon])
y_sous = np.asarray(y_fit[idx_sous_echantillon])

print(f"Sous-échantillon d'entraînement : {X_sous.shape}, "
      f"{(y_sous==1).sum()} positifs / {(y_sous==0).sum()} négatifs "
      f"({y_sous.mean():.1%} de positifs -- équilibré par construction)")


AVERTISSEMENT : seulement 107916 positifs disponibles côté fit (< 130000 demandés). N_PAR_CLASSE_REEL plafonné à 107916 -- on n'ira PAS chercher les positifs manquants dans X_val, ça fausserait l'évaluation qui suit.
Sous-échantillon d'entraînement : (215832, 189), 107916 positifs / 107916 négatifs (50.0% de positifs -- équilibré par construction)


## 4. Fonction d'évaluation (identique à `pipeline_fast_tuning_v1_4_fixed.ipynb`)

`meilleur_seuil_pr` réutilisée telle quelle (section 10 de `fast_tuning`) pour que le
protocole de sélection de seuil soit identique à celui qui a produit le F1 de
référence — pas de nouvelle méthodologie qui rendrait la comparaison bancale.


In [4]:
f1_classe1_scorer = make_scorer(f1_score, pos_label=1)


def meilleur_seuil_pr(model_predict_proba, X_val, y_val):
    """Identique à pipeline_fast_tuning_v1_4_fixed.ipynb, section 10.
    model_predict_proba : callable, ex. modele.predict_proba ou une moyenne de plusieurs."""
    probas = model_predict_proba(X_val)
    precisions, recalls, seuils = precision_recall_curve(y_val, probas)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    idx = np.argmax(f1s[:-1])
    return seuils[idx], f1s[idx]


def evaluer_sur_val(nom, probas_val, y_val):
    seuil, _ = meilleur_seuil_pr(lambda _X: probas_val, None, y_val)
    preds = (probas_val >= seuil).astype(int)
    f1_final = f1_score(y_val, preds, pos_label=1)
    print(f"\n{nom} -- seuil retenu={seuil:.3f}  F1 classe 1 (val réel)={f1_final:.4f}")
    print(classification_report(y_val, preds, target_names=["0", "1"]))
    return {"config": nom, "seuil": float(seuil), "f1_classe1_val": float(f1_final)}


## 5. Variante 1 — sous-échantillonnage simple (reproduit le test de ton ami)

Un seul tirage 130K/130K, `RandomizedSearchCV` sur une petite grille -- l'entraînement
est rapide ici (dataset ~260K lignes au lieu de 2,5M), donc `n_iter` peut être plus
généreux que dans `fast_tuning` sans risque mémoire.

**Pas de `scale_pos_weight`** : l'échantillon est déjà équilibré 50/50 par
construction, en ajouter un reviendrait à sur-corriger dans la mauvaise direction.


In [5]:
ENTRAINER_SIMPLE = True  # False pour recharger un run déjà fait

grille_simple = {
    "n_estimators": [200, 300, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.1],
    "num_leaves": [31, 63],
}

chemin_simple = f"{CHECKPOINT_DIR}/lgbm_simple.joblib"

if ENTRAINER_SIMPLE:
    t0 = time.time()
    base = LGBMClassifier(objective="binary", random_state=RANDOM_SEED, n_jobs=2, verbosity=-1)
    rs = RandomizedSearchCV(
        base, grille_simple, n_iter=15, scoring=f1_classe1_scorer, cv=3,
        random_state=RANDOM_SEED, n_jobs=1, verbose=1,
    )
    rs.fit(X_sous, y_sous)
    modele_simple = rs.best_estimator_
    print(f"Recherche terminée en {time.time()-t0:.0f}s -- meilleurs paramètres : {rs.best_params_}")
    joblib.dump(modele_simple, chemin_simple)
else:
    if not os.path.exists(chemin_simple):
        raise RuntimeError(
            f"Aucun checkpoint trouvé pour le sous-échantillonnage simple ({chemin_simple}). "
            f"Mettez ENTRAINER_SIMPLE=True pour l'entraîner une première fois."
        )
    modele_simple = joblib.load(chemin_simple)
    print(f"Modèle rechargé depuis {chemin_simple} (pas de refit)")

probas_simple = modele_simple.predict_proba(np.asarray(X_val))[:, 1]
resultat_simple = evaluer_sur_val("Sous-échantillonnage simple (130K/130K)", probas_simple, np.asarray(y_val))


Fitting 3 folds for each of 15 candidates, totalling 45 fits
Recherche terminée en 3037s -- meilleurs paramètres : {'num_leaves': 63, 'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.1}

Sous-échantillonnage simple (130K/130K) -- seuil retenu=0.745  F1 classe 1 (val réel)=0.2272
              precision    recall  f1-score   support

           0       0.97      0.93      0.95    607063
           1       0.18      0.32      0.23     27011

    accuracy                           0.91    634074
   macro avg       0.57      0.63      0.59    634074
weighted avg       0.93      0.91      0.92    634074



## 6. Variante 2 — Balanced Bagging (EasyEnsemble)

`K` modèles, chacun entraîné sur **tous** les positifs disponibles + un tirage
*différent* de négatifs de même taille, puis les probabilités moyennées. Chaque fit
individuel reste aussi léger que la variante 1 (même taille d'échantillon) — la seule
différence est qu'on répète l'opération `K` fois avec des négatifs différents à
chaque fois, ce qui permet de couvrir une bien plus grande partie des ~2,4M négatifs
sans jamais les charger tous en même temps.

Chaque sous-modèle est sauvegardé individuellement : si le notebook s'interrompt, ce
qui est déjà entraîné n'est pas reperdu (même logique que les interrupteurs
`ENTRAINER_*` de `fast_tuning`).


In [6]:
K_MODELES = 5
ENTRAINER_ENSEMBLE = True

# Meilleurs hyperparamètres trouvés en variante 1 -- réutilisés tels quels pour
# chaque sous-modèle de l'ensemble, plutôt que de relancer une recherche complète
# 5 fois (le but ici est de tester l'effet du ré-échantillonnage répété, pas de
# re-tuner à chaque fold).
meilleurs_params = modele_simple.get_params()
params_ensemble = {
    k: meilleurs_params[k]
    for k in ["n_estimators", "max_depth", "learning_rate", "num_leaves"]
}
print(f"Hyperparamètres réutilisés pour chaque sous-modèle : {params_ensemble}")

sous_modeles = []
probas_val_par_modele = []

for k in range(K_MODELES):
    chemin_k = f"{CHECKPOINT_DIR}/lgbm_ensemble_{k}.joblib"

    if ENTRAINER_ENSEMBLE and not os.path.exists(chemin_k):
        rng_k = np.random.RandomState(RANDOM_SEED + k)  # seed différente -> tirage de négatifs différent à chaque k
        idx_neg_k = rng_k.choice(idx_neg_dispo, size=N_PAR_CLASSE_REEL, replace=False)
        idx_k = np.concatenate([idx_pos_dispo, idx_neg_k])  # tous les positifs, à chaque fois
        rng_k.shuffle(idx_k)

        X_k, y_k = np.asarray(X_fit[idx_k]), np.asarray(y_fit[idx_k])
        modele_k = LGBMClassifier(
            objective="binary", random_state=RANDOM_SEED + k, n_jobs=2, verbosity=-1,
            **params_ensemble,
        )
        modele_k.fit(X_k, y_k)
        joblib.dump(modele_k, chemin_k)
        print(f"Sous-modèle {k+1}/{K_MODELES} entraîné et sauvegardé -> {chemin_k}")
        del X_k, y_k
    else:
        if not os.path.exists(chemin_k):
            raise RuntimeError(
                f"Aucun checkpoint trouvé pour le sous-modèle {k} ({chemin_k}). "
                f"Mettez ENTRAINER_ENSEMBLE=True pour l'entraîner une première fois."
            )
        modele_k = joblib.load(chemin_k)
        print(f"Sous-modèle {k+1}/{K_MODELES} rechargé depuis {chemin_k} (pas de refit)")

    sous_modeles.append(modele_k)
    probas_val_par_modele.append(modele_k.predict_proba(np.asarray(X_val))[:, 1])

probas_ensemble = np.mean(probas_val_par_modele, axis=0)
resultat_ensemble = evaluer_sur_val(f"Balanced Bagging ({K_MODELES} modèles)", probas_ensemble, np.asarray(y_val))


Hyperparamètres réutilisés pour chaque sous-modèle : {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.1, 'num_leaves': 63}
Sous-modèle 1/5 entraîné et sauvegardé -> ./models_checkpoint_undersampling/lgbm_ensemble_0.joblib
Sous-modèle 2/5 entraîné et sauvegardé -> ./models_checkpoint_undersampling/lgbm_ensemble_1.joblib
Sous-modèle 3/5 entraîné et sauvegardé -> ./models_checkpoint_undersampling/lgbm_ensemble_2.joblib
Sous-modèle 4/5 entraîné et sauvegardé -> ./models_checkpoint_undersampling/lgbm_ensemble_3.joblib
Sous-modèle 5/5 entraîné et sauvegardé -> ./models_checkpoint_undersampling/lgbm_ensemble_4.joblib

Balanced Bagging (5 modèles) -- seuil retenu=0.738  F1 classe 1 (val réel)=0.2317
              precision    recall  f1-score   support

           0       0.97      0.94      0.95    607063
           1       0.18      0.32      0.23     27011

    accuracy                           0.91    634074
   macro avg       0.58      0.63      0.59    634074
weighted avg       

## 7. Comparaison finale — tout sur le même `X_val`/`y_val` réel

In [7]:
comparaison = pd.DataFrame([
    {"config": "Référence -- scale_pos_weight (population complète)", "seuil": SEUIL_BASELINE, "f1_classe1_val": F1_BASELINE},
    resultat_simple,
    resultat_ensemble,
]).sort_values("f1_classe1_val", ascending=False).reset_index(drop=True)

comparaison


,config,seuil,f1_classe1_val
0,Balanced Bagging (5 modèles),0.738364,0.231682
1,Référence -- scale_pos_weight (population comp...,0.597000,0.228800
2,Sous-échantillonnage simple (130K/130K),0.745243,0.227197


## 8. Comment lire le résultat

- **Si les trois scores sont proches (à quelques points près)** : le rééquilibrage
  n'est pas le levier qui manquait — `scale_pos_weight` faisait déjà correctement le
  travail, et l'effort est mieux investi ailleurs (features, cf. le plan
  d'amélioration).
- **Si "Sous-échantillonnage simple" est nettement au-dessus de "Balanced Bagging"** :
  méfiance — c'est le signe classique d'un résultat qui dépend beaucoup du tirage
  aléatoire précis des 130K négatifs (variance élevée, ~96% des négatifs jamais vus).
  Le chiffre de l'ensemble (moyenné sur 5 tirages) est le plus fiable des deux.
- **Si "Balanced Bagging" dépasse nettement la référence** : c'est un vrai signal —
  le rééquilibrage agressif aide, et ça vaut la peine de l'intégrer proprement dans
  `pipeline_fast_tuning_v1_4_fixed.ipynb` (remplacer ou compléter `scale_pos_weight`
  par cette approche, avec un `K_MODELES` plus grand une fois confirmé).
- Dans tous les cas, ce F1 reste mesuré sur le même déséquilibre réel (~4,25% de
  positifs) que la référence -- directement comparable, contrairement à un F1 mesuré
  sur un jeu de test lui-même rééquilibré.
